# Efficient Subgraph Retrieval for Knowledge Graph-Enhanced LLMs
**CS 255 | Venkat Anoop Karlapudi & Jayateerth Kamatgi**

This notebook demonstrates the core pipeline:
1. Load the MetaQA Knowledge Graph
2. Explore graph statistics
3. Entity Linking  -  find a question's entity in the KG
4. BFS Subgraph Retrieval  -  collect relevant triples

---
## Step 0  -  Imports

In [ ]:
import re
import time
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque, defaultdict
from difflib import get_close_matches

# Dataset file paths
KB_PATH = '../data/raw/kb.txt'
QA_1HOP_TEST = '../data/raw/1-hop/vanilla/qa_test.txt'
QA_2HOP_TEST = '../data/raw/2-hop/vanilla/qa_test.txt'
QA_3HOP_TEST = '../data/raw/3-hop/vanilla/qa_test.txt'

print('Imports OK')

---
## Step 1  -  Load the Knowledge Graph

The KG is stored in `kb.txt`. Each line is one triple: `subject|relation|object`.

This step loads the KG into two structures:
- **NetworkX DiGraph**  -  for graph traversal algorithms
- **Adjacency list**  -  a plain Python dict for fast lookups

In [ ]:
def load_triples(kb_path):
    """Read kb.txt and return a list of (head, relation, tail) tuples."""
    triples = []
    with open(kb_path, 'r') as f:
        for line in f:
            parts = line.strip().split('|')
            if len(parts) == 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples


def build_graph(triples):
    """Build a NetworkX directed graph from triples."""
    G = nx.DiGraph()
    for head, relation, tail in triples:
        G.add_edge(head, tail, relation=relation)
    return G


def build_adjacency_list(triples):
    """
    Build a plain adjacency list.
    adj[entity] = list of (relation, neighbor) tuples
    """
    adj = defaultdict(list)
    for head, relation, tail in triples:
        adj[head].append((relation, tail))
    return dict(adj)


# --- Load everything ---
print('Loading knowledge graph...')
start = time.time()

triples = load_triples(KB_PATH)
G       = build_graph(triples)
adj     = build_adjacency_list(triples)
all_entities = set(G.nodes())

elapsed = time.time() - start
print(f'Done in {elapsed:.2f}s')

---
## Step 2  -  Graph Statistics

This section summarizes the size and structure of the KG.

In [ ]:
# Basic counts
num_triples  = len(triples)
num_nodes    = G.number_of_nodes()
num_edges    = G.number_of_edges()
all_relations = list({r for _, r, _ in triples})
avg_out_degree = num_edges / num_nodes

print('=' * 45)
print('        MetaQA Knowledge Graph Stats')
print('=' * 45)
print(f'  Total triples  : {num_triples:,}')
print(f'  Unique nodes   : {num_nodes:,}')
print(f'  Unique edges   : {num_edges:,}')
print(f'  Avg out-degree : {avg_out_degree:.2f}')
print(f'  Relations ({len(all_relations)}) :')
for r in sorted(all_relations):
    print(f'    - {r}')
print('=' * 45)

In [ ]:
# How many triples does each relation type have?
relation_counts = defaultdict(int)
for _, relation, _ in triples:
    relation_counts[relation] += 1

relations_sorted = sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)

labels  = [r for r, _ in relations_sorted]
counts  = [c for _, c in relations_sorted]

plt.figure(figsize=(10, 4))
bars = plt.barh(labels, counts, color='steelblue')
plt.xlabel('Number of Triples')
plt.title('Triple Count per Relation Type')
plt.tight_layout()
for bar, count in zip(bars, counts):
    plt.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
             f'{count:,}', va='center', fontsize=9)
plt.show()

In [ ]:
# Top 10 most connected movie nodes (highest out-degree)
out_degrees = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)

print('Top 10 most connected entities in the KG:')
print(f'  {"Entity":<40} {"Out-Degree"}')
print('  ' + '-' * 52)
for entity, degree in out_degrees[:10]:
    print(f'  {entity:<40} {degree}')

---
## Step 3  -  Entity Linking

Given a MetaQA question like `"what films were directed by [Clint Eastwood]"`, this step:
1. **Extracts** the entity mention from the `[brackets]`
2. **Matches** it to an actual node in the KG

Extraction uses bracket text. Matching uses exact lookup first, then fuzzy matching as a fallback.

In [ ]:
def extract_entity(question):
    """Pull the entity mention out of [brackets] in a MetaQA question."""
    match = re.search(r'\[(.+?)\]', question)
    if match:
        return match.group(1)
    return None


def link_entity(mention, all_entities):
    """
    Map a mention string to a node in the KG.
    Strategy:
      1. Exact match
      2. Case-insensitive exact match
      3. Fuzzy match (difflib)
    Returns (matched_entity, confidence_score)
    """
    # 1. Exact match
    if mention in all_entities:
        return mention, 1.0

    # 2. Case-insensitive match
    lower_map = {e.lower(): e for e in all_entities}
    if mention.lower() in lower_map:
        return lower_map[mention.lower()], 0.99

    # 3. Fuzzy match
    close = get_close_matches(mention, all_entities, n=1, cutoff=0.6)
    if close:
        return close[0], 0.75

    return None, 0.0


# Example questions for a small sanity check
test_questions = [
    "what movies are about [ginger rogers]",
    "what films were directed by [Clint Eastwood]",
    "which movies can be described by [moore]",
    "who acted in [The Dark Knight]",
]

print(f'  {"Question":<55} {"Linked Entity":<30} {"Confidence"}')
print('  ' + '-' * 100)
for q in test_questions:
    mention = extract_entity(q)
    entity, score = link_entity(mention, all_entities)
    print(f'  {q:<55} {str(entity):<30} {score:.2f}')

---
## Step 4  -  BFS Subgraph Retrieval

**Breadth-First Search (BFS)** performs hop-by-hop graph traversal from a seed entity and collects encountered triples.

- `k=1` -> direct neighbors only (typical for 1-hop questions)
- `k=2` -> neighbors of neighbors (typical for 2-hop questions)
- `k=3` -> three levels deep (typical for 3-hop questions)
- `max_triples` -> hard cap to keep the subgraph a manageable size

In [ ]:
def bfs_subgraph(graph, seed_entity, k=2, max_triples=50):
    """BFS-based subgraph retrieval (bidirectional traversal).

    Note: In MetaQA, many edges point from Movie -> (Actor/Director/etc.).
    Bidirectional traversal allows traversal from any entity type.
    """
    if seed_entity not in graph:
        return []

    visited = {seed_entity}
    queue = deque([(seed_entity, 0)])  # (node, hop)
    result = []

    while queue:
        node, hop = queue.popleft()
        if hop >= k:
            continue

        # Outgoing edges: node -> neighbor
        for neighbor in graph.neighbors(node):
            relation = graph[node][neighbor]["relation"]
            result.append((node, relation, neighbor))
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, hop + 1))
            if len(result) >= max_triples:
                return result

        # Incoming edges: predecessor -> node
        for predecessor in graph.predecessors(node):
            relation = graph[predecessor][node]["relation"]
            result.append((predecessor, relation, node))
            if predecessor not in visited:
                visited.add(predecessor)
                queue.append((predecessor, hop + 1))
            if len(result) >= max_triples:
                return result

    return result


print("BFS function defined.")

In [ ]:
# Example BFS run
seed = 'Clint Eastwood'

for k in [1, 2]:
    start = time.time()
    subgraph = bfs_subgraph(G, seed, k=k, max_triples=100)
    elapsed  = (time.time() - start) * 1000  # ms

    print(f'\nBFS k={k} from "{seed}"  ->  {len(subgraph)} triples  ({elapsed:.2f} ms)')
    print('  First 5 triples:')
    for head, rel, tail in subgraph[:5]:
        print(f'    ({head}) --[{rel}]--> ({tail})')

---
## Step 5  -  End-to-End Pipeline

This section runs the complete pipeline:
**Question -> Entity Extraction -> Entity Linking -> BFS Retrieval -> Triples Output**

Example questions are taken from each hop level of the MetaQA test set.

In [ ]:
def run_pipeline(question, graph, all_entities, k=2, max_triples=50):
    """
    Full pipeline for a single question.
    Returns a dict with all intermediate results and timing.
    """
    t0 = time.time()

    # Step 1: Extract entity mention from question
    mention = extract_entity(question)

    # Step 2: Link mention to KG node
    t1 = time.time()
    entity, confidence = link_entity(mention, all_entities)
    link_time = (time.time() - t1) * 1000

    # Step 3: BFS subgraph retrieval
    t2 = time.time()
    subgraph = bfs_subgraph(graph, entity, k=k, max_triples=max_triples) if entity else []
    retrieval_time = (time.time() - t2) * 1000

    return {
        'question'      : question,
        'mention'       : mention,
        'entity'        : entity,
        'confidence'    : confidence,
        'subgraph'      : subgraph,
        'link_time_ms'  : link_time,
        'retrieval_ms'  : retrieval_time,
    }


def print_result(result, ground_truth=None, show_triples=5):
    print('\n' + '=' * 65)
    print(f'  Question  : {result["question"]}')
    if ground_truth:
        print(f'  Answer(s) : {ground_truth}')
    print(f'  Mention   : {result["mention"]}')
    print(f'  KG Entity : {result["entity"]}  (confidence: {result["confidence"]:.2f})')
    print(f'  Triples   : {len(result["subgraph"])} retrieved')
    print(f'  Timing    : linking={result["link_time_ms"]:.1f}ms  retrieval={result["retrieval_ms"]:.1f}ms')
    print(f'  Sample triples:')
    for head, rel, tail in result['subgraph'][:show_triples]:
        print(f'    ({head}) --[{rel}]--> ({tail})')
    print('=' * 65)


print('Pipeline functions defined.')

In [ ]:
# Load a few real test questions from each hop level
def load_qa(path, n=3):
    """Load the first n (question, answers) pairs from a QA file."""
    samples = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            parts = line.strip().split('\t')
            if len(parts) == 2:
                samples.append((parts[0], parts[1]))
    return samples

samples_1hop = load_qa(QA_1HOP_TEST, n=2)
samples_2hop = load_qa(QA_2HOP_TEST, n=2)
samples_3hop = load_qa(QA_3HOP_TEST, n=2)

print('Loaded sample questions.')

In [ ]:
# --- 1-Hop Questions (k=1) ---
print('\n### 1-HOP QUESTIONS  (BFS k=1) ###')
for question, answers in samples_1hop:
    result = run_pipeline(question, G, all_entities, k=1)
    print_result(result, ground_truth=answers)

In [ ]:
# --- 2-Hop Questions (k=2) ---
print('\n### 2-HOP QUESTIONS  (BFS k=2) ###')
for question, answers in samples_2hop:
    result = run_pipeline(question, G, all_entities, k=2)
    print_result(result, ground_truth=answers)

In [ ]:
# --- 3-Hop Questions (k=3) ---
print('\n### 3-HOP QUESTIONS  (BFS k=3) ###')
for question, answers in samples_3hop:
    result = run_pipeline(question, G, all_entities, k=3, max_triples=100)
    print_result(result, ground_truth=answers)

---
## Step 6  -  Subgraph Answer Coverage (Recall Check)

This section checks whether the retrieved subgraph **contains** the ground truth answer entities.
This is a simple retrieval-quality metric: **Answer Recall**.

- **Recall = 1.0** -> all answer entities appeared in the retrieved triples
- **Recall = 0.0** -> the subgraph missed the answer entirely

In [ ]:
def compute_recall(subgraph, ground_truth_str):
    """
    Check what fraction of ground truth answers appear in the retrieved triples.
    ground_truth_str: pipe-separated string like 'Clint Eastwood|Steven Spielberg'
    """
    answers = set(ground_truth_str.strip().split('|'))
    # Collect all entity names that appear in the subgraph
    subgraph_entities = set()
    for head, _, tail in subgraph:
        subgraph_entities.add(head)
        subgraph_entities.add(tail)

    hits = answers & subgraph_entities
    recall = len(hits) / len(answers) if answers else 0.0
    return recall, hits


# Evaluate recall on all loaded samples
all_samples = [
    (1, samples_1hop, 1),
    (2, samples_2hop, 2),
    (3, samples_3hop, 3),
]

print(f'  {"Hop":<5} {"Question":<55} {"Recall":<8} {"Hits"}')
print('  ' + '-' * 100)

recall_by_hop = defaultdict(list)

for hop, samples, k in all_samples:
    for question, answers in samples:
        result = run_pipeline(question, G, all_entities, k=k, max_triples=200)
        recall, hits = compute_recall(result['subgraph'], answers)
        recall_by_hop[hop].append(recall)
        short_q = question[:52] + '...' if len(question) > 52 else question
        print(f'  {hop:<5} {short_q:<55} {recall:<8.2f} {hits}')

print()
for hop in [1, 2, 3]:
    avg = sum(recall_by_hop[hop]) / len(recall_by_hop[hop])
    print(f'  Average Recall @ {hop}-hop : {avg:.2f}')

---
## Step 7  -  Subgraph Size vs Hops

As k increases, the subgraph grows rapidly. This plot shows the **tradeoff** between coverage and size.

In [ ]:
# Pick a representative seed entity
seed = 'Clint Eastwood'

hop_values    = [1, 2, 3]
subgraph_sizes = []
retrieval_times = []

for k in hop_values:
    t = time.time()
    sg = bfs_subgraph(G, seed, k=k, max_triples=10000)
    elapsed = (time.time() - t) * 1000
    subgraph_sizes.append(len(sg))
    retrieval_times.append(elapsed)
    print(f'  k={k}: {len(sg)} triples in {elapsed:.1f} ms')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar([str(k) for k in hop_values], subgraph_sizes, color='steelblue')
ax1.set_xlabel('BFS Hops (k)')
ax1.set_ylabel('Number of Triples Retrieved')
ax1.set_title(f'Subgraph Size vs Hops\n(seed: "{seed}")')
for i, v in enumerate(subgraph_sizes):
    ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')

ax2.bar([str(k) for k in hop_values], retrieval_times, color='darkorange')
ax2.set_xlabel('BFS Hops (k)')
ax2.set_ylabel('Retrieval Time (ms)')
ax2.set_title(f'Retrieval Time vs Hops\n(seed: "{seed}")')
for i, v in enumerate(retrieval_times):
    ax2.text(i, v + 0.5, f'{v:.1f}ms', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Summary

| Component | Status |
|---|---|
| Knowledge Graph loaded |  134,741 triples, 43,234 entities |
| Entity Linking |  Exact + fuzzy matching |
| BFS Subgraph Retrieval |  Configurable k-hops |
| End-to-end pipeline |  1-hop, 2-hop, 3-hop questions |
| Answer Recall metric |  Checks if answers appear in subgraph |